# Figure 2 — per-dataset paired F1 differences (N-LDF)

**Gaya grafik OriginLab, dijalankan penuh di Google Colab.**

### Kenapa bukan paket `originpro`

Semua paket Python dari OriginLab — `originpro`, `OriginExt`, `OrglabExt` —
berjalan **hanya di Windows** dan menuntut instalasi lokal **Origin 2021 atau
lebih baru**. Halaman PyPI `originpro` menyatakannya langsung:

> *This package is Windows only and requires a local installation of Origin 2021 or later.*

Google Colab berjalan di Linux tanpa Origin. Dukungan teknis OriginLab menjawab
pertanyaan yang sama di forum mereka dengan: *"Origin HAS to be installed on the
same machine on which the code executes."* Jadi `pip install originpro` di Colab
akan terpasang, tetapi `import originpro` selalu gagal — tidak ada COM server
Origin untuk disambungkan.

Notebook ini menempuh jalan yang benar-benar berjalan di Colab: **mereproduksi
gaya grafik bawaan Origin dengan matplotlib** — bingkai kotak penuh, tick major
dan minor mengarah ke dalam pada keempat sisi, font Arial, tanpa grid, legenda
berbingkai. Sel terakhir juga mengekspor CSV rapi supaya Anda bisa menggambar
ulang di Origin asli kalau mau.


In [ ]:
# 1 — font Arial. Colab sudah punya Liberation Sans (metrik identik dengan
#     Arial); baris di bawah menambahkan Arial asli kalau tersedia.
!apt-get -qq install -y fonts-liberation > /dev/null 2>&1

import matplotlib
import matplotlib.font_manager as _fm
try:                                   # muat ulang daftar font setelah apt
    _fm.fontManager = _fm._load_fontmanager(try_read_cache=False)
except AttributeError:                 # matplotlib lama
    _fm.fontManager.__init__()
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
from matplotlib.ticker import AutoMinorLocator

avail = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
print('Arial tersedia        :', 'Arial' in avail)
print('Liberation Sans ada   :', 'Liberation Sans' in avail)


In [ ]:
# 2 — data. Jalankan salah satu dari tiga cara di bawah.
import os

CSV = 'dataset_method_means.csv'

# (a) sudah dipush ke GitHub -> baca langsung, tanpa unggah
# CSV = ('https://raw.githubusercontent.com/ahmadilham-idsrg/n-ldf/'
#        'main/results/dataset_method_means.csv')

# (b) dari Google Drive
# from google.colab import drive; drive.mount('/content/drive')
# CSV = '/content/drive/MyDrive/n-ldf/results/dataset_method_means.csv'

# (c) unggah manual
if not str(CSV).startswith('http') and not os.path.exists(CSV):
    from google.colab import files
    print('Pilih dataset_method_means.csv dari komputer Anda:')
    CSV = list(files.upload())[0]

piv = pd.read_csv(CSV).pivot(index='Dataset', columns='Method', values='F1')
print(piv.shape[0], 'dataset ×', piv.shape[1], 'metode')
piv.head(3)


In [ ]:
# 3 — gaya Origin sebagai rcParams matplotlib
BLACK = '#000000'

def origin_style():
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans'],
        'font.size': 9,
        'axes.linewidth': 1.0,          # bobot bingkai Origin
        'axes.edgecolor': BLACK, 'axes.labelcolor': BLACK,
        'axes.grid': False,             # Origin tidak menggambar grid
        'text.color': BLACK,
        'xtick.color': BLACK, 'ytick.color': BLACK,
        'xtick.direction': 'in', 'ytick.direction': 'in',
        'xtick.top': True, 'ytick.right': True,
        'xtick.major.size': 5.0, 'xtick.minor.size': 2.6,
        'ytick.major.size': 5.0, 'ytick.minor.size': 2.6,
        'xtick.major.width': 1.0, 'xtick.minor.width': 0.8,
        'ytick.major.width': 1.0, 'ytick.minor.width': 0.8,
        'legend.frameon': True, 'legend.fancybox': False,
        'legend.edgecolor': BLACK, 'legend.framealpha': 1.0,
        'legend.borderpad': 0.45,
        'figure.facecolor': 'white', 'axes.facecolor': 'white',
        'savefig.facecolor': 'white',
        'pdf.fonttype': 42, 'ps.fonttype': 42,   # TrueType, bukan Type 3
    })

# Pasangan warna divergen, sudah divalidasi untuk buta warna:
# ΔE deuteranopia 21,6 · penglihatan normal 32,3 · kontras ≥3:1 di atas putih.
# Merah #FF0000 dan biru #0000FF bawaan Origin sengaja tidak dipakai karena
# keduanya menyatu di bawah deuteranopia.
POS, NEG, TIE = '#2A78D6', '#E34948', '#898781'


In [ ]:
# 4 — gambar
ROWS = [('ADASYN', 'ADASYN'), ('B-SMOTE', 'Borderline-SMOTE'),
        ('SMOTE', 'SMOTE'), ('SMOTE-Tomek', 'SMOTE-Tomek'),
        ('Baseline', 'No resampling')]
YPOS = {'ADASYN': 4.0, 'B-SMOTE': 3.0, 'SMOTE': 2.0,
        'SMOTE-Tomek': 1.0, 'Baseline': -0.35}

origin_style()
rng = np.random.default_rng(42)      # jitter tetap -> gambar reproducible
fig, ax = plt.subplots(figsize=(5.9, 3.15))

for key, label in ROWS:
    diff = (piv['N-LDF'] - piv[key]).dropna().values
    y = YPOS[key]
    jit = rng.uniform(-0.21, 0.21, size=diff.size)
    col = np.where(diff > 1e-9, POS, np.where(diff < -1e-9, NEG, TIE))
    ax.scatter(diff, y + jit, s=25, c=col, alpha=0.85,
               edgecolors='white', linewidths=0.8, zorder=4, clip_on=False)
    med = float(np.median(diff))
    ax.plot([med, med], [y - 0.40, y + 0.40], color=BLACK, lw=2.0,
            solid_capstyle='butt', zorder=6)
    w = int((diff > 1e-9).sum()); l = int((diff < -1e-9).sum())
    t = int(diff.size - w - l)
    tr = ax.get_yaxis_transform()
    ax.text(1.030, y, f'{w}\u2013{t}\u2013{l}', transform=tr, va='center',
            ha='left', fontsize=8.5,
            fontweight='bold' if key == 'Baseline' else 'normal')
    ax.text(1.190, y, f'{med:+.4f}'.replace('-', '\u2212'), transform=tr,
            va='center', ha='left', fontsize=8)

ax.axvline(0, color='#9A9A9A', lw=0.9, ls=(0, (4, 2.5)), zorder=2)
ax.set_yticks([YPOS[k] for k, _ in ROWS])
ax.set_yticklabels([lab for _, lab in ROWS], fontsize=9)
ax.set_ylim(-1.05, 4.75); ax.set_xlim(-0.28, 0.42)
ax.set_xticks(np.arange(-0.2, 0.41, 0.1))
ax.xaxis.set_minor_locator(AutoMinorLocator(2))
ax.tick_params(axis='y', which='both', length=0)
ax.tick_params(axis='x', which='major', labelsize=8.5)
ax.set_xlabel('F1 difference per dataset:  N-LDF $-$ comparison strategy',
              fontsize=9, labelpad=5)

tr = ax.get_yaxis_transform()
ax.text(1.030, 4.60, 'win\u2013tie\u2013loss', transform=tr, va='center',
        ha='left', fontsize=7.5)
ax.text(1.190, 4.60, 'median', transform=tr, va='center', ha='left',
        fontsize=7.5)
ax.plot([-0.28, 0.42], [0.35, 0.35], color='#B4B4B4', lw=0.8, zorder=3)

h = [plt.Line2D([], [], marker='o', ls='', ms=5.0, mfc=c, mec='white', mew=0.8)
     for c in (POS, TIE, NEG)]
h.append(plt.Line2D([], [], color=BLACK, lw=1.8))
ax.legend(h, ['N-LDF higher', 'tie', 'comparison higher', 'median'],
          loc='lower center', bbox_to_anchor=(0.44, -0.335), ncol=4,
          fontsize=8, handletextpad=0.45, columnspacing=1.5)

for ext in ('pdf', 'png', 'tif'):
    fig.savefig(f'fig_paired_diff_origin.{ext}', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# 5 — angka yang tercetak di gambar, untuk dicocokkan dengan naskah
rows = []
for key, label in ROWS:
    d = (piv['N-LDF'] - piv[key]).dropna()
    rows.append({'Comparison': label, 'n': len(d),
                 'win': int((d > 1e-9).sum()),
                 'tie': int((d.abs() <= 1e-9).sum()),
                 'loss': int((d < -1e-9).sum()),
                 'median': round(float(d.median()), 4),
                 'min': round(float(d.min()), 4),
                 'max': round(float(d.max()), 4)})
pd.DataFrame(rows)


In [ ]:
# 6 — CSV rapi untuk digambar ulang di Origin asli (Windows)
tidy = []
for key, label in ROWS:
    d = (piv['N-LDF'] - piv[key]).dropna()
    for ds, v in d.items():
        tidy.append({'Comparison': label, 'Dataset': ds,
                     'Diff_F1': round(float(v), 6),
                     'Outcome': 'Win' if v > 1e-9 else
                                ('Loss' if v < -1e-9 else 'Tie')})
tidy = pd.DataFrame(tidy)
tidy.to_csv('fig_paired_diff_origin_data.csv', index=False)
print(len(tidy), 'baris ditulis')
tidy.head()


In [ ]:
# 7 — unduh hasilnya
from google.colab import files
for f in ('fig_paired_diff_origin.pdf', 'fig_paired_diff_origin.png',
          'fig_paired_diff_origin.tif', 'fig_paired_diff_origin_data.csv'):
    files.download(f)


---

## Kalau Anda ingin menggambarnya di Origin sungguhan

Sel 6 menulis `fig_paired_diff_origin_data.csv` dalam format panjang
(`Comparison`, `Dataset`, `Diff_F1`, `Outcome`) — 205 baris, satu per titik.
Di komputer Windows dengan Origin 2021+:

1. **File → Import → Single ASCII**, pilih CSV itu.
2. Set kolom: `Diff_F1` sebagai **X**, dan tambahkan satu kolom indeks baris
   sebagai **Y** (nilai 4, 3, 2, 1, −0.35 untuk kelima baris perbandingan).
3. **Plot → Categorical → Scatter**, dengan `Comparison` sebagai variabel
   pengelompokan dan `Outcome` sebagai indeks warna.
4. Tambahkan jitter vertikal lewat **Plot Details → Spacing → Offset** atau
   kolom jitter yang dihitung sendiri.
5. Median per baris: tambahkan sebagai plot kedua bertipe **Drop Line**.

Alternatif yang jauh lebih singkat: jalankan `make_fig_paired_origin.py` di
komputer Anda — hasilnya identik dengan notebook ini dan tidak perlu Origin
sama sekali.
